Lab05

Bai01

In [1]:
# -*- coding: utf-8 -*-
"""Buoc 1: tu tinh Gini va chon cau hoi chia tot nhat."""
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text

nho = pd.DataFrame({
    "thu_nhap": [6, 8, 9, 11, 13, 15, 18, 20, 25, 30],
    "co_viec_on_dinh": [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
    "tra_dung_han": [0, 1, 0, 0, 1, 1, 0, 1, 1, 1],
})
y = nho["tra_dung_han"].to_numpy()

def gini(nhan):
    """Chi so Gini cua mot nut hai lop."""
    if len(nhan) == 0:
        return 0.0
    p1 = np.mean(nhan)
    return 1 - p1**2 - (1 - p1)**2

def gini_sau_chia(di_trai, y):
    """Gini trung binh co trong so cua hai nhanh."""
    trai = y[di_trai]
    phai = y[~di_trai]
    n = len(y)
    return len(trai) / n * gini(trai) + len(phai) / n * gini(phai)

g_goc = gini(y)
print(f"Gini cua nut goc: {g_goc:.4f}")
print()

thu = nho["thu_nhap"].to_numpy()
muc = np.sort(np.unique(thu))
print("Cau hoi 'thu_nhap <= nguong ?'")
print(" nguong   Gini sau chia   Gini giam")
for a, b in zip(muc[:-1], muc[1:]):
    nguong = (a + b) / 2
    g = gini_sau_chia(thu <= nguong, y)
    print(f" {nguong:6.1f} {g:13.4f} {g_goc - g:12.4f}")
print()

viec = nho["co_viec_on_dinh"].to_numpy()
g_viec = gini_sau_chia(viec == 0, y)
print("Cau hoi 'co viec on dinh ?'")
print(f" Gini sau chia {g_viec:.4f}, Gini giam {g_goc - g_viec:.4f}")
print()

cot = ["thu_nhap", "co_viec_on_dinh"]
cay = DecisionTreeClassifier(max_depth=1, random_state=0)
cay.fit(nho[cot], y)
print("Cay mot tang do scikit-learn chon:")
print(export_text(cay, feature_names=cot, show_weights=True))

Gini cua nut goc: 0.4800

Cau hoi 'thu_nhap <= nguong ?'
 nguong   Gini sau chia   Gini giam
    7.0        0.4000       0.0800
    8.5        0.4750       0.0050
   10.0        0.4190       0.0610
   12.0        0.3167       0.1633
   14.0        0.4000       0.0800
   16.5        0.4500       0.0300
   19.0        0.3429       0.1371
   22.5        0.4000       0.0800
   27.5        0.4444       0.0356

Cau hoi 'co viec on dinh ?'
 Gini sau chia 0.4190, Gini giam 0.0610

Cay mot tang do scikit-learn chon:
|--- thu_nhap <= 12.00
|   |--- weights: [3.00, 1.00] class: 0
|--- thu_nhap >  12.00
|   |--- weights: [1.00, 5.00] class: 1



Bai02

In [3]:
# -*- coding: utf-8 -*-
"""Buoc 2: hoc cay 5 la tren du lieu khach vay."""
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

df = pd.read_csv("khach_vay.csv")
print("Kich thuoc bang:", df.shape)
print(df.head())
print("Ty le khach tra dung han:", round(df["tra_dung_han"].mean(), 2))

X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

cay = DecisionTreeClassifier(max_leaf_nodes=5, random_state=0)
cay.fit(X_tr, y_tr)

print(f"Do sau {cay.get_depth()}, so la {cay.get_n_leaves()}")
print(export_text(cay, feature_names=list(X.columns),
                  show_weights=True, decimals=3))

a_hoc = accuracy_score(y_tr, cay.predict(X_tr))
a_kt = accuracy_score(y_te, cay.predict(X_te))
print(f"Accuracy tap hoc      : {a_hoc:.4f}")
print(f"Accuracy tap kiem tra : {a_kt:.4f}")
print()

moi = pd.DataFrame({
    "thu_nhap": [20.0, 10.0],
    "ty_le_no": [0.4, 0.3],
    "co_viec_on_dinh": [1, 0],
    "so_nguoi_phu_thuoc": [2, 1],
})
du_doan = cay.predict(moi)
xac_suat = cay.predict_proba(moi)[:, 1]
for i in range(len(moi)):
    ket = "dung han" if du_doan[i] == 1 else "tre han"
    print(f"Khach {i + 1}: {ket}, xac suat dung han {xac_suat[i]:.2f}")

Kich thuoc bang: (600, 5)
   thu_nhap  ty_le_no  co_viec_on_dinh  so_nguoi_phu_thuoc  tra_dung_han
0       9.4      0.18                0                   4             0
1      15.8      0.70                0                   2             1
2       6.1      0.79                0                   0             0
3      19.4      1.13                1                   4             0
4      24.8      0.33                1                   0             1
Ty le khach tra dung han: 0.49
Do sau 3, so la 5
|--- thu_nhap <= 15.050
|   |--- ty_le_no <= 0.615
|   |   |--- co_viec_on_dinh <= 0.500
|   |   |   |--- weights: [33.000, 6.000] class: 0
|   |   |--- co_viec_on_dinh >  0.500
|   |   |   |--- weights: [35.000, 47.000] class: 1
|   |--- ty_le_no >  0.615
|   |   |--- weights: [114.000, 14.000] class: 0
|--- thu_nhap >  15.050
|   |--- ty_le_no <= 1.020
|   |   |--- weights: [27.000, 145.000] class: 1
|   |--- ty_le_no >  1.020
|   |   |--- weights: [20.000, 9.000] class: 0

Accura

Bai03

In [5]:
# -*- coding: utf-8 -*-
"""Buoc 3: cay moc tu do, chon do sau bang cross-validation."""
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

tu_do = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
print("Cay moc tu do")
print(f" do sau {tu_do.get_depth()}, so la {tu_do.get_n_leaves()}")
a_hoc = accuracy_score(y_tr, tu_do.predict(X_tr))
a_kt = accuracy_score(y_te, tu_do.predict(X_te))
print(f" tap hoc {a_hoc:.4f}, tap kiem tra {a_kt:.4f}")
print()

print("Do sau   Tap hoc   Kiem dinh cheo")
diem = {}
for d in range(1, 16):
    mh = DecisionTreeClassifier(max_depth=d, random_state=0)
    diem[d] = cross_val_score(mh, X_tr, y_tr, cv=5).mean()
    hoc = accuracy_score(y_tr, mh.fit(X_tr, y_tr).predict(X_tr))
    print(f" {d:2d} {hoc:12.4f} {diem[d]:15.4f}")

d_tot = max(diem, key=diem.get)
print()
print(f"Do sau tot nhat = {d_tot}")

tot = DecisionTreeClassifier(max_depth=d_tot, random_state=0)
tot.fit(X_tr, y_tr)
print(f"Cay co {tot.get_n_leaves()} la")
acc = accuracy_score(y_te, tot.predict(X_te))
print(f"Accuracy tren tap kiem tra = {acc:.4f}")

Cay moc tu do
 do sau 13, so la 92
 tap hoc 1.0000, tap kiem tra 0.7667

Do sau   Tap hoc   Kiem dinh cheo
  1       0.7467          0.7244
  2       0.7711          0.7378
  3       0.8000          0.7511
  4       0.8644          0.8089
  5       0.8978          0.8333
  6       0.9133          0.8156
  7       0.9400          0.8067
  8       0.9600          0.7733
  9       0.9800          0.7800
 10       0.9911          0.7822
 11       0.9933          0.7756
 12       0.9978          0.7778
 13       1.0000          0.7733
 14       1.0000          0.7733
 15       1.0000          0.7733

Do sau tot nhat = 5
Cay co 27 la
Accuracy tren tap kiem tra = 0.8533


Bai04

In [7]:
# -*- coding: utf-8 -*-
"""Buoc 4: kiem chung cay quyet dinh khong can chuan hoa."""
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

cay_tho = DecisionTreeClassifier(max_depth=5, random_state=0)
cay_tho.fit(X_tr, y_tr)

cay_chuan_hoa = make_pipeline(
    StandardScaler(),
    DecisionTreeClassifier(max_depth=5, random_state=0)
)
cay_chuan_hoa.fit(X_tr, y_tr)

pred_tho = cay_tho.predict(X_te)
pred_chuan = cay_chuan_hoa.predict(X_te)

print(f"Accuracy cay tren du lieu tho        : {accuracy_score(y_te, pred_tho):.4f}")
print(f"Accuracy cay sau StandardScaler     : {accuracy_score(y_te, pred_chuan):.4f}")
print(f"So khach du doan khac nhau          : {np.sum(pred_tho != pred_chuan)}")
print("Ket luan: cay chi so sanh nguong nen khong can chuan hoa.")

Accuracy cay tren du lieu tho        : 0.8533
Accuracy cay sau StandardScaler     : 0.8533
So khach du doan khac nhau          : 0
Ket luan: cay chi so sanh nguong nen khong can chuan hoa.


Bai05

In [8]:
# -*- coding: utf-8 -*-
"""Buoc 5: trong rung ngau nhien va so voi cay don."""
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

cay = DecisionTreeClassifier(max_depth=5, random_state=0)
cay.fit(X_tr, y_tr)

rung = RandomForestClassifier(
    n_estimators=200,
    random_state=0,
    n_jobs=-1
)
rung.fit(X_tr, y_tr)

for ten, mo_hinh in [("Cay don", cay), ("Rung ngau nhien", rung)]:
    a_hoc = accuracy_score(y_tr, mo_hinh.predict(X_tr))
    a_kt = accuracy_score(y_te, mo_hinh.predict(X_te))
    print(f"{ten:16s} | tap hoc {a_hoc:.4f} | tap kiem tra {a_kt:.4f}")

Cay don          | tap hoc 0.8978 | tap kiem tra 0.8533
Rung ngau nhien  | tap hoc 1.0000 | tap kiem tra 0.9000


Bai06

In [9]:
# -*- coding: utf-8 -*-
"""Buoc 6: doc do quan trong dac trung cua rung."""
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

df = pd.read_csv("khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

rung = RandomForestClassifier(
    n_estimators=300,
    random_state=0,
    n_jobs=-1
)
rung.fit(X_tr, y_tr)

imp = pd.Series(rung.feature_importances_, index=X.columns)
imp = imp.sort_values(ascending=False)
print("Do quan trong theo Gini cua rung:")
print(imp.round(4))
print()

perm = permutation_importance(
    rung, X_te, y_te,
    n_repeats=10,
    random_state=0,
    n_jobs=-1
)
perm_s = pd.Series(perm.importances_mean, index=X.columns)
perm_s = perm_s.sort_values(ascending=False)
print("Permutation importance tren tap kiem tra:")
print(perm_s.round(4))
print()
print("Gioi han: neu hai dac trung tuong quan manh, do quan trong co the bi chia se")
print("cho nhau, nen khong nen doc mot con so qua tuyet doi.")

Do quan trong theo Gini cua rung:
thu_nhap              0.4829
ty_le_no              0.3374
so_nguoi_phu_thuoc    0.0920
co_viec_on_dinh       0.0877
dtype: float64

Permutation importance tren tap kiem tra:
thu_nhap              0.3120
ty_le_no              0.1753
co_viec_on_dinh       0.0993
so_nguoi_phu_thuoc    0.0420
dtype: float64

Gioi han: neu hai dac trung tuong quan manh, do quan trong co the bi chia se
cho nhau, nen khong nen doc mot con so qua tuyet doi.


In [10]:
# -*- coding: utf-8 -*-
"""Buoc 7: cay hoi quy tren du lieu gia nha."""
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text

df = pd.read_csv("gia_nha.csv")

# Tuy du lieu bai 1: neu cot gia ten khac, sua lai cho dung.
target_col = "gia" if "gia" in df.columns else df.columns[-1]
X = df.drop(columns=target_col)
y = df[target_col]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17
)

cay = DecisionTreeRegressor(max_depth=4, random_state=0)
cay.fit(X_tr, y_tr)

pred = cay.predict(X_te)
rmse = np.sqrt(mean_squared_error(y_te, pred))
print(f"Cot muc tieu: {target_col}")
print(f"RMSE tren tap kiem tra: {rmse:.4f}")
print()
print(export_text(cay, feature_names=list(X.columns), decimals=2))

Cot muc tieu: gia
RMSE tren tap kiem tra: 0.6964

|--- dien_tich <= 73.90
|   |--- dien_tich <= 52.10
|   |   |--- tuoi_nha <= 14.00
|   |   |   |--- so_phong <= 1.50
|   |   |   |   |--- value: [3.99]
|   |   |   |--- so_phong >  1.50
|   |   |   |   |--- value: [3.87]
|   |   |--- tuoi_nha >  14.00
|   |   |   |--- tuoi_nha <= 18.00
|   |   |   |   |--- value: [4.60]
|   |   |   |--- tuoi_nha >  18.00
|   |   |   |   |--- value: [4.25]
|   |--- dien_tich >  52.10
|   |   |--- dien_tich <= 68.55
|   |   |   |--- tuoi_nha <= 22.00
|   |   |   |   |--- value: [5.20]
|   |   |   |--- tuoi_nha >  22.00
|   |   |   |   |--- value: [4.56]
|   |   |--- dien_tich >  68.55
|   |   |   |--- tuoi_nha <= 7.00
|   |   |   |   |--- value: [6.21]
|   |   |   |--- tuoi_nha >  7.00
|   |   |   |   |--- value: [5.55]
|--- dien_tich >  73.90
|   |--- dien_tich <= 101.55
|   |   |--- dien_tich <= 89.70
|   |   |   |--- tuoi_nha <= 6.00
|   |   |   |   |--- value: [8.11]
|   |   |   |--- tuoi_nha >  6.00
